# 02 — Feature Engineering: The Wearable-Extractable Feature Set

This notebook handles data cleaning and feature engineering for both layers of the project.

**Layer 2 — Apple Watch bridge layer**
- **Calls:** `src/preprocess.py`
- **Input:** `data/apple_watch/*_raw.csv`
- **Output:** `data/processed/*_clean.csv`

**Layer 1 — Physionet clinical benchmark**
- **Calls:** `src/features.py`
- **Input:** `data/physionet/*.mat` + `REFERENCE-v3.csv`
- **Output:** `data/processed/physionet_features.csv`

## Cleaning Pipeline — `src/preprocess.py`

Runs `run_cleaning_pipeline()` across all raw Apple Watch CSVs. For each metric the pipeline:

1. **Parses dates** — converts `startDate` to datetime, extracts `hour` and `day_of_week`
2. **Drops non-numeric values** — coerces the `value` column; any non-castable entries become NaN and are removed
3. **Applies physiological threshold filters** — removes readings outside clinically plausible bounds (e.g., heart rate 30–220 bpm, SpO₂ 70–100%, etc.)
4. **Assigns anchor periods** — labels each record relative to the June 18, 2025 anchor date (baseline, pre_anchor, post_anchor, follow_up)
5. **Saves cleaned CSVs** to `data/processed/`

The summary table below shows retention rates per metric. Expect >99% retention — most removals are non-numeric parse failures, not outliers.

In [ ]:
import sys
import os
import subprocess
import pandas as pd

PROJECT_ROOT = subprocess.check_output(
    ['git', 'rev-parse', '--show-toplevel'], text=True
).strip()
sys.path.insert(0, PROJECT_ROOT)

from src.preprocess import run_cleaning_pipeline

RAW_DIR  = os.path.join(PROJECT_ROOT, 'data', 'apple_watch')
PROC_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

print("="*65)
print("APPLE WATCH DATA CLEANING PIPELINE")
print("02_feature_engineering.ipynb")
print("="*65)

reports = run_cleaning_pipeline(RAW_DIR, PROC_DIR)

print(f"\n{'METRIC':<22} {'INITIAL':>10} "
      f"{'REMOVED':>10} {'FINAL':>10} {'RETAINED':>10}")
print("-"*65)

for r in reports:
    removed = r['non_numeric'] + r['outliers_removed']
    print(f"{r['metric']:<22} "
          f"{r['initial_records']:>10,} "
          f"{removed:>10,} "
          f"{r['final_records']:>10,} "
          f"{r['retention_pct']:>9.1f}%")

print("="*65)
print("\nCleaned files saved to data/processed/")
print("Anchor date: June 18 2025")
print("\nAnchor period definitions:")
print("  baseline    \u2014 more than 91 days before anchor")
print("  pre_anchor  \u2014 within 90 days before anchor")
print("  post_anchor \u2014 within 90 days after anchor")
print("  follow_up   \u2014 more than 90 days after anchor")

## Post-Pipeline Artifact Removal — 210 bpm Sensor Spike

The cleaning pipeline uses a 30–220 bpm threshold filter, so the 210 bpm reading at `2021-08-28 01:50:38+08:00` passed through uncaught. Context from edge case inspection:

| Time | Value | Context |
|------|-------|---------|
| 01:49:53 | 39 bpm | Sleep reading |
| 01:49:55 | 39 bpm | Sleep reading |
| 01:50:38 | **210 bpm** | **Artifact** — 171 bpm jump in 43 seconds during sleep |

A 171 bpm spike within 43 seconds during sleep is physiologically impossible. This is a confirmed sensor artifact (likely watch repositioning). We remove it directly from `heart_rate_clean.csv`.

**Calls:** nothing from `src/` — targeted removal only

In [ ]:
import pandas as pd
import os

PROC_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

hr_path = os.path.join(PROC_DIR, 'heart_rate_clean.csv')
hr = pd.read_csv(hr_path)

initial = len(hr)

# Remove confirmed sensor artifact
# 210 bpm at 01:50 AM following two 39 bpm sleep readings
# 171 bpm jump in 60 seconds during sleep — physiologically impossible
artifact_mask = (
    (hr['startDate'] == '2021-08-28 01:50:38+08:00') &
    (hr['value'] == 210.0)
)

hr = hr[~artifact_mask]
hr.to_csv(hr_path, index=False)

print(f"Artifact removal complete.")
print(f"Records before: {initial:,}")
print(f"Records after:  {len(hr):,}")
print(f"Removed:        {initial - len(hr)}")

## Quick Check — Heart Rate Distribution & Anchor Periods

Inspection only — no `src/` calls. Validates that cleaned heart rate data has a realistic distribution and confirms anchor period sample sizes for Layer 2 analysis.

In [ ]:
import pandas as pd
import os

PROC_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

hr = pd.read_csv(os.path.join(PROC_DIR, 'heart_rate_clean.csv'))

print("Heart Rate Distribution Check")
print("="*40)
print(f"Min:    {hr['value'].min():.1f} bpm")
print(f"Max:    {hr['value'].max():.1f} bpm")
print(f"Mean:   {hr['value'].mean():.1f} bpm")
print(f"Median: {hr['value'].median():.1f} bpm")
print(f"Std:    {hr['value'].std():.1f} bpm")
print(f"\nReadings below 40 bpm: "
      f"{(hr['value'] < 40).sum():,}")
print(f"Readings above 200 bpm: "
      f"{(hr['value'] > 200).sum():,}")
print(f"\nAnchor period distribution:")
print(hr['anchor_period'].value_counts().to_string())

In [6]:
edge_cases = hr[
    (hr['value'] < 40) | (hr['value'] > 200)
][['startDate', 'value', 'hour', 'anchor_period']]

print("Edge Case Readings")
print("="*55)
print(edge_cases.to_string())

Edge Case Readings
                        startDate  value  hour anchor_period
38248   2021-08-28 01:49:53+08:00   39.0     1      baseline
38249   2021-08-28 01:49:55+08:00   39.0     1      baseline
436885  2025-12-09 20:38:25+08:00  203.0    20     follow_up
436886  2025-12-09 20:38:29+08:00  205.0    20     follow_up
436887  2025-12-09 20:38:35+08:00  203.0    20     follow_up
436888  2025-12-09 20:38:37+08:00  202.0    20     follow_up
436889  2025-12-09 20:38:41+08:00  201.0    20     follow_up


### Why These Eight Features?

The feature set is not arbitrary — it is **constrained by what consumer wearables can actually measure**.

A model that uses features only extractable from clinical-grade equipment cannot be tested on wearable data. The generalisation test becomes impossible by design. So the constraint is the methodology: every feature must be derivable from both a single-lead clinical ECG and a consumer wearable PPG signal.

**Excluded — permanently:** Frequency domain features (LF/HF ratio, HF power). These require raw beat-by-beat intervals at high, consistent sampling rates — unavailable from consumer wearables.

**Selected — eight time-domain and statistical features:** RMSSD, SDNN, Mean RR, pNN50, HR Mean, HR Std Dev, RR Skewness, RR Kurtosis. All eight are computable from RR intervals derived from either ECG R-peaks or PPG peaks. The same function will be applied to Physionet ECG in this notebook, Apple Watch data in notebook 04, and MIMIC PERform AF PPG in notebook 05.

The feature set is locked here. It cannot be changed between layers.

## Feature Extraction — `src/features.py`

Extracts the eight locked HRV and RR interval features from 8,249 Physionet 2017 clinical ECG recordings (after excluding 279 noisy-class records). These features are constrained to measurements that consumer wearables can produce, enabling valid Layer 1 → Layer 2 generalisation.

**Locked feature set** (frequency domain excluded):
- RMSSD, SDNN, Mean RR, pNN50
- HR Mean, HR Std Dev, RR Skewness, RR Kurtosis

**Pipeline steps per recording:**
1. Read raw ECG waveform via `wfdb.rdrecord()`
2. Detect R-peaks using XQRS detector
3. Convert R-peak locations to RR intervals (ms)
4. Quality filter: 300–2000ms bounds, minimum 10 valid intervals
5. Compute eight feature values

**Binary label mapping:** N → 0 (Normal), A/O → 1 (Abnormal)

**Calls:** `src/features.build_feature_matrix()`
**Input:** `data/physionet/*.mat` + `data/physionet/REFERENCE-v3.csv`
**Output:** `data/processed/physionet_features.csv`

In [ ]:
import sys
import os
import pandas as pd

from src.features import build_feature_matrix

PHYSIONET_DIR = os.path.join(PROJECT_ROOT, 'data', 'physionet')
LABELS_PATH   = os.path.join(PROJECT_ROOT, 'data', 'physionet', 'REFERENCE-v3.csv')
PROC_DIR      = os.path.join(PROJECT_ROOT, 'data', 'processed')

# ── Run extraction pipeline ──────────────────────────────────────
df = build_feature_matrix(PHYSIONET_DIR, LABELS_PATH)

# ── Inspect result ───────────────────────────────────────────────
print("\nFeature Matrix — First 5 Rows")
print("="*55)
print(df.head().to_string())

print("\nMissing Value Check")
print("="*55)
missing = df.isnull().sum()
print(missing.to_string())
if missing.sum() == 0:
    print("\nNo missing values — feature matrix is complete.")
else:
    print(f"\nWARNING: {missing.sum()} missing values detected.")
    print("Do not proceed to modelling until resolved.")

print(f"\nFinal shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# ── Save to processed directory ──────────────────────────────────
out_path = os.path.join(PROC_DIR, 'physionet_features.csv')
df.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path}")

In [ ]:
import pandas as pd
import os

PROC_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
df = pd.read_csv(os.path.join(PROC_DIR, 'physionet_features.csv'))

print("Missing Value Check")
print("="*40)
print(df.isnull().sum().to_string())

print(f"\nClass Distribution")
print("="*40)
print(f"Normal   (0): {(df['binary_label']==0).sum():,} "
      f"({(df['binary_label']==0).mean()*100:.1f}%)")
print(f"Abnormal (1): {(df['binary_label']==1).sum():,} "
      f"({(df['binary_label']==1).mean()*100:.1f}%)")

print(f"\nFeature Summary Statistics")
print("="*40)
feature_cols = ['rmssd','sdnn','mean_rr','pnn50',
                'hr_mean','hr_std','rr_skewness','rr_kurtosis']
print(df[feature_cols].describe().round(2).to_string())

### Features Ready
- **Physionet feature matrix:** 8,187 records, 8 features, zero missing values.
- **Class distribution:** 61.6% Normal (5,042), 38.4% Abnormal (3,145).

**Next:** Notebook 03 trains four candidate models on these features and selects one for deployment. The selection criteria are pre-registered and set before training begins.